## `Загрузка данных (1 балл)`

In [1]:
!pip install yadisk

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.7/140.7 kB 11.7 MB/s eta 0:00:00


In [2]:
TOKEN = "y0__wgBEOqqzbcEGIesRyC5wf_SGDDNzNLrB1UKpR0S6zv56ylh9XeIO-G_bAgG"

In [3]:
# ============================================
# БЛОК 2: Загрузка и РАСПАКОВКА АРХИВА
# ============================================

import yadisk
import zipfile
import io
import os
from tqdm import tqdm
import concurrent.futures
import requests

y = yadisk.YaDisk(token=TOKEN)

# Получаем ссылку на скачивание
print("Получение ссылки на скачивание...")
download_link = y.get_download_link("test_stage1.zip")
print(f"✅ Ссылка получена")

# Скачиваем архив на диск (НЕ В ПАМЯТЬ!)
print("Скачивание архива на диск...")
response = requests.get(download_link, stream=True)
total_size = int(response.headers.get('content-length', 0))

# Скачиваем в файл на диске
temp_zip = "temp_archive.zip"
with open(temp_zip, "wb") as f:
    with tqdm(total=total_size, unit='B', unit_scale=True, desc="Скачивание") as pbar:
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)
            pbar.update(len(chunk))

print(f"✅ Архив скачан на диск, размер: {os.path.getsize(temp_zip) / (1024**2):.2f} МБ")

# Распаковываем на диск (НЕ В ПАМЯТЬ!)
extract_dir = "extracted_files"
os.makedirs(extract_dir, exist_ok=True)

print("Распаковка на диск...")
with zipfile.ZipFile(temp_zip, 'r') as zip_ref:
    file_list = [f for f in zip_ref.infolist() if not f.is_dir()]
    print(f"Распаковка {len(file_list)} файлов...")

    for file_info in tqdm(file_list, desc="Распаковка"):
        zip_ref.extract(file_info, extract_dir)

print(f"✅ Распаковано {len(file_list)} файлов на диск")

# Удаляем временный архив
os.remove(temp_zip)
print("✅ Временный архив удален")

Получение ссылки на скачивание...
✅ Ссылка получена
Скачивание архива на диск...


Скачивание: 100%|██████████| 592M/592M [00:54<00:00, 10.8MB/s]


✅ Архив скачан на диск, размер: 564.14 МБ
Распаковка на диск...
Распаковка 2162 файлов...


Распаковка: 100%|██████████| 2162/2162 [00:02<00:00, 775.43it/s]


✅ Распаковано 2162 файлов на диск
✅ Временный архив удален


In [4]:
import os

import numpy as np
import numpy.testing as npt
import pandas as pd
import torch
from torchvision.io import read_image
from torch.utils.data import Dataset, DataLoader
from torchvision.models import vgg13, VGG13_Weights
import torch
from sklearn.model_selection import train_test_split

from PIL import Image
from torchvision import transforms
import matplotlib_inline
import matplotlib.pyplot as plt

%matplotlib inline
matplotlib_inline.backend_inline.set_matplotlib_formats('pdf', 'svg')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [5]:
import pandas as pd
import os

# Читаем CSV
df = pd.read_csv('/content/extracted_files/test_stage1/test.csv')

# Базовая папка, где лежит test_stage1_img
base_dir = '/content/extracted_files/test_stage1/'

# Список полных путей к изображениям
full_paths = [os.path.join(base_dir, path) for path in df['img_path']]

# Проверка
print(f"Найдено {len(full_paths)} изображений")
print("Первый путь:", full_paths[0])

Найдено 2160 изображений
Первый путь: /content/extracted_files/test_stage1/test_stage1_img/35105be77490f7c573e5edee94e311e9.jpg


## `U-Net (3 балла)`

#### `Encoder`

In [6]:
from torch import nn
from copy import deepcopy
class VGG13Encoder(torch.nn.Module):
    def __init__(self, num_blocks, weights=VGG13_Weights.DEFAULT):
        super().__init__()
        self.num_blocks = num_blocks

        # Будем использовать предобученную VGG13 в качестве backbone
        feature_extractor = vgg13(weights=weights).features

        # Каждый блок энкодера U-Net — это блок VGG13 без MaxPool2d
        self.blocks = torch.nn.ModuleList()
        tmp=[]
        cur=0
        for idx in range(self.num_blocks):
            # Возьмите нужные слои из `feature_extractor` для очередного U-Net блока
            # Объедините их с помощью `torch.nn.Sequential`
            tmp=[]
            while(not isinstance(feature_extractor[cur],nn.MaxPool2d)):
              tmp.append(deepcopy(feature_extractor[cur]))
              cur+=1
            self.blocks.append(
                  nn.Sequential(*tmp)
            )
            cur+=1
        self.MP=nn.MaxPool2d(kernel_size=2, stride=2, padding=0)


    def forward(self, x):
        activations = []
        for idx, block in enumerate(self.blocks):
            # Примените очередной блок U-Net
            # your code here
            x=block(x)

            # Сохраните активации для передачи их в декодер
            # your code here
            activations.append(x)

            # При необходимости примените max-pool
            # Можно использовать `torch.functional.F.max_pool2d`
            # your code here
            x=self.MP(x) if idx!=len(self.blocks)-1 else x

        return activations

In [7]:
assert sum((param.numel() for param in VGG13Encoder(num_blocks=1).parameters())) == 38720
assert sum((param.numel() for param in VGG13Encoder(num_blocks=2).parameters())) == 260160
assert sum((param.numel() for param in VGG13Encoder(num_blocks=3).parameters())) == 1145408

x = torch.arange(1 * 3 * 320 * 240).reshape(1, 3, 320, 240) / (1 * 3 * 320 * 240)
out = VGG13Encoder(num_blocks=3)(x)

assert len(out) == 3
npt.assert_almost_equal(torch.linalg.norm(out[0]).item(), 902.218, decimal=3)
npt.assert_almost_equal(torch.linalg.norm(out[1]).item(), 571.030, decimal=3)
npt.assert_almost_equal(torch.linalg.norm(out[2]).item(), 648.068, decimal=3)

Downloading: "https://download.pytorch.org/models/vgg13-19584684.pth" to /root/.cache/torch/hub/checkpoints/vgg13-19584684.pth


100%|██████████| 508M/508M [00:07<00:00, 75.2MB/s]


#### `Decoder`

In [8]:
import torch.nn.functional as F
class DecoderBlock(torch.nn.Module):
    def __init__(self, out_channels):
        super().__init__()

        self.upconv = torch.nn.Conv2d(
            in_channels=out_channels * 2, out_channels=out_channels,
            kernel_size=3, padding=1, dilation=1
        )
        self.conv1 = torch.nn.Conv2d(
            in_channels=out_channels * 2, out_channels=out_channels,
            kernel_size=3, padding=1, dilation=1
        )
        self.conv2 = torch.nn.Conv2d(
            in_channels=out_channels, out_channels=out_channels,
            kernel_size=3, padding=1, dilation=1
        )
        self.relu = torch.nn.ReLU()

    def forward(self, down, left):
        # Upsample x2 и свёртка
        # your code here
        x = F.interpolate(down, scale_factor=2, mode='nearest')
        x=self.upconv(x)

        # Конкатенация выхода энкодера и предыдущего блока декодера
        # your code here
        x = torch.cat([x,left],dim=1)

        # Две свёртки с ReLu
        # your code here
        x = self.conv1(x)
        x=self.relu(x)
        x = self.conv2(x)
        x=self.relu(x)

        return x

In [9]:
class Decoder(torch.nn.Module):
    def __init__(self, num_filters, num_blocks):
        super().__init__()

        self.blocks = torch.nn.ModuleList()
        for idx in range(num_blocks):
            self.blocks.insert(0, DecoderBlock(num_filters * 2 ** idx))

    def forward(self, acts):
        up = acts[-1]
        for block, left in zip(self.blocks, acts[-2::-1]):
            up = block(up, left)
        return up

#### `U-Net`

In [10]:
class UNet(torch.nn.Module):
    def __init__(self, num_classes=1, num_blocks=4):
        super().__init__()
        # your code here
        self.encoder = VGG13Encoder(num_blocks)

        # your code here
        self.decoder = Decoder(64,num_blocks-1)

        # Свёртка 1x1 для попиксельной агрегации каналов
        # your code here
        self.final = torch.nn.Conv2d(
            in_channels=64, out_channels=num_classes,
            kernel_size=1,
        )

    def forward(self, x):
        x=self.encoder(x)
        x=self.decoder(x)
        x=self.final(x)

        return x

In [11]:
model = UNet(num_classes=1, num_blocks=3)
x = torch.arange(1 * 3 * 320 * 240).reshape(1, 3, 320, 240) / (1 * 3 * 320 * 240)
assert sum((param.numel() for param in model.parameters())) == 2067649
assert list(model(x).shape) == [1, 1, 320, 240]
model

UNet(
  (encoder): VGG13Encoder(
    (blocks): ModuleList(
      (0): Sequential(
        (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (1): ReLU(inplace=True)
        (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (3): ReLU(inplace=True)
      )
      (1): Sequential(
        (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (1): ReLU(inplace=True)
        (2): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (3): ReLU(inplace=True)
      )
      (2): Sequential(
        (0): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (1): ReLU(inplace=True)
        (2): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (3): ReLU(inplace=True)
      )
    )
    (MP): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (decoder): Decoder(
    (blocks): ModuleList(
      (0): DecoderBlock(
        

### `Оценивание качества сети`


Обычно, оптимизируемый функционал сложно интерпретировать, а так же в разных экспериментах могут использоваться разные функции потерь. Поэтому необходимо замерять и отслеживать независимую метрику качества. Предлагается использовать **IoU (Intersection over Union)** — один из стандартных вариантов для задачи сегментации:
$$
\text{IoU} = \frac{|A \cap B|}{|A \cup B|} = \frac{\sum\limits_{ij}a_{ij}b_{ij}}{\sum\limits_{ij}a_{ij} + b_{ij} - a_{ij}b_{ij}}
$$

Учтите, что знаменатель может быть равен нулю, например, если маска нулевая. Такие объекты можно не учитывать при агрегации и пропускать, так как в ином случае подсчет метрики на всем батче будет NaN. В pytorch есть встроенные для этого функции с префиксом nan.


In [12]:
class IoUScore(torch.nn.Module):
    def __init__(self, threshold, reduction=None,with_logits=True):
        """
        Arguments
        ---------
        threshold : float
            threshold for logits binarization
        reduction : Optional[str] (None, 'mean' or 'sum')
            specifies the reduction to apply to the output:

            None: no reduction will be applied
            'mean': the sum of the output will be divided by the number of elements in the batch
            'sum':  the output will be summed.
        with_logits : bool
            If True, use additional sigmoid for inputs
        """
        super().__init__()

        self.threshold = threshold
        self.reduction = reduction
        self.with_logits=with_logits
    @torch.no_grad()
    def forward(self, logits, true_labels):
        """
        Arguments
        ---------
        logits: torch.Tensor
            Unnormalized probability of true class. Shape: [B, ...]
        true_labels: torch.Tensor
            Mask of correct predictions. Shape: [B, ...]
        Returns
        -------
        torch.Tensor
            If reduction is 'mean' or 'sum' returns a tensor with a single element
            Otherwise, returns a tensor of shape [B]
        """
        if(self.with_logits):
          logits=torch.sigmoid(logits)
        logits=(logits>self.threshold).float()
        axis=[i for i in range(1,logits.ndim)]
        a=(logits*true_labels).sum(dim=axis)
        b=logits.sum(dim=axis)
        c=true_labels.sum(dim=axis)
        score=a/(b+c-a)

        if self.reduction == 'sum':
            # your code here
            score = torch.nansum(score)
        elif self.reduction == 'mean':
            # your code here
            score = torch.nanmean(score)

        return score

In [13]:
logits = torch.tensor([
    [
        [0.3089,  0.4311, -0.9711],
        [0.9030,  1.0325, -0.7607],
        [0.9648, -0.5528, -1.1010]
    ], [
        [0.8906,  0.8099,  0.4458],
        [2.6215, -1.3198,  0.3142],
        [0.2262, -0.9175, -0.0174]
    ], [
        [0.8906,  0.8099,  0.4458],
        [2.6215, -1.3198,  0.3142],
        [0.2262, -0.9175, -0.0174]
    ]
])
true_labels = torch.tensor([
    [
        [0., 1., 0.],
        [1., 0., 0.],
        [0., 0., 1.]
    ], [
        [1., 1., 1.],
        [1., 0., 1.],
        [1., 0., 0.]
    ], [
        [1., 1., 1.],
        [0., 1., 1.],
        [1., 1., 1.]
    ]
])

scores = IoUScore(0.0,with_logits=False)(logits, true_labels)
npt.assert_almost_equal(scores.numpy(), np.array([0.3333, 1.0000, 0.5556]), decimal=4)

score = IoUScore(0.0, reduction='sum',with_logits=False)(logits, true_labels)
npt.assert_almost_equal(score.item(), 1.8889, decimal=4)

score = IoUScore(0.0, reduction='mean',with_logits=False)(logits, true_labels)
npt.assert_almost_equal(score.item(), 0.6296, decimal=4)

### `Логирование`

При реализации цикла обучения необходимо сохранять метрики $1\text{-}9$ с использованием модуля [torch.utils.tensorboard](https://pytorch.org/docs/stable/tensorboard.html) или с помощью модуля [wandb](https://docs.wandb.ai/tutorials/) вам предлагается самим выбрать более удобный для вас модуль.

Для отслеживания процесса обучения обычно требуется сохранять информацию разных его аспектах.

Для полного контроля над процессом обучения обычно требуется сохранять информацию о разных статистиках. Самыми очевидными являются:
1. Значение функции потерь (лосса) на обучающей и тестовой выборках
2. Метики качества, например, **Dice Coefficient** и **IoU (Intersection over Union)** на обучающей и тестовой выборках

Однако, суррогатные метрики могут не отражать реального качества. Поэтому для моделей, которые выдают визуализируемый результат, обычно логируют предсказания для некоторых объектов выборки. В задаче сегментации естественным вариантом является логирование четвёрки: исходное изображение, истинная маска, маска вероятностей истинного класса, бинаризованная маска.

*Замечание:* лосс и качество на обучающей выборке обычно логируют не только в конце каждой эпохи, но и по отдельным батчам на каждой итерации.

3. Чтобы отслеживать динамику обучения необходимо зафиксировать небольшой набор объектов обучающей и тестовой выборок и после каждой эпохи обучения логировать указанные четыре картинки для каждого объекта.

*Замечание:* можно логировать четвёрки изображений независимо, однако, удобнее объединить их на одной фигуре [реализация через Tensorboard](https://pytorch.org/docs/stable/tensorboard.html#torch.utils.tensorboard.writer.SummaryWriter.add_figure), [реализация через Wandb](https://docs.wandb.ai/guides/track/log/plots/#matplotlib-and-plotly-plots) (лучше делать через fig и Image). Дополнительный плюс — возможность подписать значения метрик для этого объекта в заголовке изображения или добавить colorbar для более простой интерпретации предсказанной маски.

Для удобной категоризации экспериментов обычно в начале обучения сохраняют:

4. Гиперпараметры модели [реализация через Tensorboard](https://pytorch.org/docs/stable/tensorboard.html#torch.utils.tensorboard.writer.SummaryWriter.add_hparams), [реализация через Wandb](https://docs.wandb.ai/guides/track/config/)
5. Структуру модели [реализация через Tensorboard](https://pytorch.org/docs/stable/tensorboard.html#torch.utils.tensorboard.writer.SummaryWriter.add_graph), **про реализацию wandb ниже**

Для исследования технических особенностей обучения полезно логировать следующие статистики:

6. Распределение весов, активаций, градиентов [реализация через Tensorboard](https://pytorch.org/docs/stable/tensorboard.html#torch.utils.tensorboard.writer.SummaryWriter.add_histogram), [реализация через Wandb](https://docs.wandb.ai/ref/python/data-types/histogram/). Сохранять гистограммы каждого параметра после каждой итерации может быть вычислительно неэффективно, поэтому обычно сохраняют распределения весов для каждого отдельного слоя нейронной сети после каждой эпохи.
7. Норма весов и норма градиента на каждой итерации

Наконец, после каждой эпохи можно визуализировать промежуточные представления входных данных:

8. Активации после каждого слоя/блока, как изображения
9. Градиенты функции потерь по активациям для некоторых объектов обучающей выборки, как изображения

*Замечание:* для реализации пунктов 8, 9 менять код модели не требуется. Используйте хуки: [register_full_backward_hook](https://pytorch.org/docs/stable/generated/torch.nn.Module.html#torch.nn.Module.register_full_backward_hook) (данный хук не работает с inplace операциями, например, с `torch.nn.ReLU(inplace=True)`. Или не используйте inplace операции, или используйте [`Tensor.register_hook`](https://github.com/pytorch/pytorch/issues/61519)), [register_forward_hook](https://pytorch.org/docs/stable/generated/torch.nn.Module.html#torch.nn.Module.register_forward_hook).

Файлы с логами, а также чекпоинты весов модели после итерации с наилучшим валидационным качеством (смотрите [torch.save](https://pytorch.org/docs/stable/generated/torch.save.html) и [torch.nn.Module.state_dict](https://pytorch.org/tutorials/recipes/recipes/what_is_state_dict.html)) для **ВСЕХ** проведённых экспериментов (за исключением упавших, недосчитанных и так далее) необходимо сдать в anytask.

#### `Некоторые советы по wandb`

- При логировании всегда указывайте параметр step, который должен не уменьшаться.

- При логировании изображений лучше их нормировать в диапазон [0, 1] как в функции `show_idx_image` выше. Для логирования предсказаний маски, лучше использовать вероятности. При неправильном диапозоне изображения могут отображаться неверно!

- Вместо логирования изображений, лучше логировать `matplotlib` графики и соответсвующие им `figure`. Кроме того, лучше закрывать все открытые графики, чтобы не перегружать ноутбук. Пример кода:


```python
fig, ax = plt.subplots(...)
...
wandb.log({key: wandb.Image(fig)}, step=global_step)
plt.close('all')
```

- Функционал построение bins в `wandb.Histogram` ограничен, можете посмотреть в сторону аргумента `np_histogram`.

- Wandb не имеет широкого функционала для логирования структуры модели. Есть способ через wandb.watch(net, log_graph=True), но получаемый результаты сильно отличаются от возможностей `tensorboard`. К счастью, `wandb` умеет синхронизировать логи с `Tensorboard` поэтому можно использовать следующую логику:


```python
wandb.init(
    project=project_name,
    name=run_name,
    config=config,
    sync_tensorboard=True        # Синхронизация с tensorboard
)

writer = SummaryWriter(run_name) # Создание логера tensorboard

# Логирование графа модели
writer.add_graph(net, input_x)
writer.close()

train_loop(...)
wandb.finish()
```

- `wandb` по умолчанию сохраняет все логи локально и на сайте в вашем личном хранилище. В случае перезагрузки `Google Colab` локальные файлы не сохраняются! Кроме того, скачивание файлов не через GoogleDisk достаточно продолжительное (в районе 20-30 минут).

- В `wandb` можно легко сохранять код и указать директорию, так вы никогда не забудете какой именно код получает результаты логов.

```python
wandb.init(..., settings=wandb.Settings(code_dir="."))
```

Кроме того, можно сохранить отдельные файлы:

```python
wandb.init(..., save_code=True)
wandb.run.log_code(
    "./",
    include_fn=lambda path: condition(path)
)
```

- В `wandb` есть функция для автоматического логирования весов и градиентов модели `wandb.watch()`, но функционал слишком ограничен и накладывает слишком много накладных расходов. **Лучше писать руками!**

#### `Некоторые советы по tensorboard`

- Функционал и возможности `tensorboard` шире (работа с 3D, продвинутые визуализации).

- `tensorboard` сохраняет логи только локально, то есть после закрытия `Google Colab` ничего не сохранится. В реальных проектах это является сильной стороной, так как гарантирует безопасность и конфиденциальность.

- Для визуализации логов достаточно запустить в ноутбуке или в терминале соответсвующее расширение, однако в случае `Google Colab` для этого приходится прокидывать порты, [подробнее тут](https://stackoverflow.com/questions/47818822/can-i-use-tensorboard-with-google-colab).

- `tensorboard` меньше лагает и более легковесный.


# код

In [15]:
import pandas as pd
import os
import torch
from PIL import Image
from torchvision import transforms
from tqdm import tqdm
import yadisk

TOKEN = "y0__wgBEOqqzbcEGIesRyC5wf_SGDDNzNLrB1UKpR0S6zv56ylh9XeIO-G_bAgG"
y = yadisk.YaDisk(token=TOKEN)

# Конфигурация
CSV_PATH = "/content/extracted_files/test_stage1/submission.csv"
TEST_BASE_DIR = "/content/extracted_files/test_stage1"
OUTPUT_CSV = "/content/submission.csv"
PREDICTIONS_DIR = "/content/predictions"
REMOTE_WEIGHTS = "/IoU/checkpoint_epoch_0.pth"
LOCAL_WEIGHTS = "/tmp/final_model.pth"
NUM_BLOCKS = 4
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Создаём папку
os.makedirs(PREDICTIONS_DIR, exist_ok=True)

# Загрузка весов с Яндекс Диска
print("Скачивание весов с Яндекс Диска...")
y.download(REMOTE_WEIGHTS, LOCAL_WEIGHTS)

# Загрузка модели
print("Загрузка модели...")
model = UNet(num_classes=1, num_blocks=NUM_BLOCKS).to(DEVICE)

#model.load_state_dict(torch.load(LOCAL_WEIGHTS, map_location=DEVICE))

checkpoint = torch.load(LOCAL_WEIGHTS, map_location=DEVICE)
model.load_state_dict(checkpoint['model_state_dict'])

model.eval()

# Удаляем локальный файл
os.remove(LOCAL_WEIGHTS)

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

def pad_to_1024(image_tensor):
    _, h, w = image_tensor.shape
    new_img = torch.zeros(3, 1024, 1024)
    new_img[:, :h, :w] = image_tensor
    return new_img

df = pd.read_csv(CSV_PATH)
results = []

for idx, row in tqdm(df.iterrows(), total=len(df), desc="Инференс"):
    img_rel_path = row['img_path']
    img_full_path = os.path.join(TEST_BASE_DIR, img_rel_path)

    img = Image.open(img_full_path).convert('RGB')
    original_size = img.size[::-1]

    img_tensor = transform(img)
    img_tensor = pad_to_1024(img_tensor).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        pred = model(img_tensor)
        pred = torch.sigmoid(pred)
        pred = (pred > 0.5).float()

    pred_resized = torch.nn.functional.interpolate(
        pred, size=original_size, mode='bilinear', align_corners=False
    )
    pred_resized = (pred_resized > 0.5).float().cpu().squeeze(0).squeeze(0)

    mask_name = os.path.basename(img_rel_path).rsplit('.', 1)[0] + '_pred.png'
    mask_path = os.path.join(PREDICTIONS_DIR, mask_name)
    mask_img = Image.fromarray((pred_resized.numpy() * 255).astype('uint8'))
    mask_img.save(mask_path)

    results.append({
        'img_path': img_rel_path,
        'prediction_path': f"predictions/{mask_name}"
    })

df_results = pd.DataFrame(results)
df_results.to_csv(OUTPUT_CSV, index=False)

print(f"✅ Готово! Сохранено {len(results)} масок")
print(f"📄 Файл соответствий: {OUTPUT_CSV}")

Скачивание весов с Яндекс Диска...
Загрузка модели...


Инференс: 100%|██████████| 2160/2160 [11:11<00:00,  3.22it/s]

✅ Готово! Сохранено 2160 масок
📄 Файл соответствий: /content/submission.csv


In [17]:
import zipfile
import os

OUTPUT_DIR = "/content"
PREDICTIONS_DIR = "/content/predictions"
CSV_PATH = "/content/submission.csv"
ZIP_PATH = "/content/submission.zip"

print("Создание архива для отправки...")

with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zipf:
    # Добавляем CSV
    if os.path.exists(CSV_PATH):
        zipf.write(CSV_PATH, os.path.basename(CSV_PATH))
        print(f"✅ Добавлен CSV: {CSV_PATH}")

    # Добавляем все маски из папки predictions
    if os.path.exists(PREDICTIONS_DIR):
        for root, dirs, files in os.walk(PREDICTIONS_DIR):
            for file in files:
                file_path = os.path.join(root, file)
                arcname = os.path.join('predictions', file)
                zipf.write(file_path, arcname)
        print(f"✅ Добавлены маски из {PREDICTIONS_DIR}")

print(f"✅ Архив создан: {ZIP_PATH}")
print(f"Размер: {os.path.getsize(ZIP_PATH) / (1024**2):.2f} МБ")

Создание архива для отправки...
✅ Добавлен CSV: /content/submission.csv
✅ Добавлены маски из /content/predictions
✅ Архив создан: /content/submission.zip
Размер: 2.81 МБ


In [ ]:
import pandas as pd
import os
import torch
from PIL import Image
from torchvision import transforms
from tqdm import tqdm
import yadisk

TOKEN = "y0__wgBEOqqzbcEGIesRyC5wf_SGDDNzNLrB1UKpR0S6zv56ylh9XeIO-G_bAgG"
y = yadisk.YaDisk(token=TOKEN)

# Конфигурация
CSV_PATH = "/content/extracted_files/test_stage1/submission.csv"
TEST_BASE_DIR = "/content/extracted_files/test_stage1"
OUTPUT_CSV = "/content/submission.csv"
PREDICTIONS_DIR = "/content/predictions"
REMOTE_WEIGHTS = "/IoU/checkpoint_epoch_0.pth"
LOCAL_WEIGHTS = "/tmp/final_model.pth"
NUM_BLOCKS = 4
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Создаём папку
os.makedirs(PREDICTIONS_DIR, exist_ok=True)

# Загрузка весов с Яндекс Диска
print("Скачивание весов с Яндекс Диска...")
y.download(REMOTE_WEIGHTS, LOCAL_WEIGHTS)

# Загрузка модели
print("Загрузка модели...")
model = UNet(num_classes=1, num_blocks=NUM_BLOCKS).to(DEVICE)

checkpoint = torch.load(LOCAL_WEIGHTS, map_location=DEVICE)

# Проверяем структуру чекпоинта
if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
    # Если есть model_state_dict - загружаем из него
    model.load_state_dict(checkpoint['model_state_dict'])
    print("✅ Чекпоинт загружен (model_state_dict)")
elif isinstance(checkpoint, dict) and all(isinstance(v, torch.Tensor) for v in checkpoint.values()):
    # Если это просто model.state_dict() (OrderedDict)
    model.load_state_dict(checkpoint)
    print("✅ Чекпоинт загружен (model.state_dict)")
else:
    print("❌ Неизвестная структура чекпоинта")

model.eval()

# Удаляем локальный файл
os.remove(LOCAL_WEIGHTS)

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

def pad_to_1024(image_tensor):
    _, h, w = image_tensor.shape
    new_img = torch.zeros(3, 1024, 1024)
    new_img[:, :h, :w] = image_tensor
    return new_img

df = pd.read_csv(CSV_PATH)
results = []

for idx, row in tqdm(df.iterrows(), total=len(df), desc="Инференс"):
    img_rel_path = row['img_path']
    img_full_path = os.path.join(TEST_BASE_DIR, img_rel_path)

    img = Image.open(img_full_path).convert('RGB')
    original_size = img.size[::-1]

    img_tensor = transform(img)
    img_tensor = pad_to_1024(img_tensor).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        pred = model(img_tensor)
        pred = torch.sigmoid(pred)
        pred = (pred > 0.5).float()

    pred_resized = torch.nn.functional.interpolate(
        pred, size=original_size, mode='bilinear', align_corners=False
    )
    pred_resized = (pred_resized > 0.5).float().cpu().squeeze(0).squeeze(0)

    mask_name = os.path.basename(img_rel_path).rsplit('.', 1)[0] + '_pred.png'
    mask_path = os.path.join(PREDICTIONS_DIR, mask_name)
    mask_img = Image.fromarray((pred_resized.numpy() * 255).astype('uint8'))
    mask_img.save(mask_path)

    results.append({
        'img_path': img_rel_path,
        'prediction_path': f"predictions/{mask_name}"
    })

df_results = pd.DataFrame(results)
df_results.to_csv(OUTPUT_CSV, index=False)

print(f"✅ Готово! Сохранено {len(results)} масок")
print(f"📄 Файл соответствий: {OUTPUT_CSV}")